# 蜃景 × lightx2v —— 纯文生视频 (t2v) Colab 部署（无 ComfyUI）

**怎么用：菜单「代码执行程序 → 全部运行」(Run all)。** 断线/回收后再 Run all——**Wan 权重(67G)持久化 Drive、不重下**；lightx2v 每会话重装（很快，`--no-deps`）、SageAttention 现编译（可选，几分钟）。

**跑前一次性：**
1. 运行时选 **GPU**（Blackwell / Hopper / Ada / A100 都行）。
2. 右侧 🔑 Secrets 可加 **HF_TOKEN**（下 Wan 权重快些；公开模型匿名也能下）。
3. 第 1 格 `DEEPSEEK_KEY` 可留空（分镜 / AI 分析用前端配的 grok 模型）。

**§3/§4 已把 2026-06-17 实战踩平的坑固化（不用再手动改）：**
- lightx2v 0.1.0 钉死 `torch<2.8.0`，撞 Colab/Blackwell 的 cu128 torch 2.8 → 本体 `pip install -e . --no-deps`、依赖也 `--no-deps` 补（绕过 torch 解析，2.8 跑得动）。
- torch↔torchvision ABI 不匹配 → 子进程探测，坏了就原子重装匹配的 cu128 三件套。
- `worldmirror` 硬 import flash_attn、`ring_attn` 的 `@torch.jit.script`（多卡才用）新 torch 编不过 → 自动 patch（改文件，server 子进程也生效）。
- editable 的 `.pth` 内核不重读 → 一切**以干净子进程验证**（§5 server 本就是子进程）。
- 权重用 `snapshot_download`（hf CLI 多个 `--include` 会被忽略 → 大权重不下）。
- 注意力默认 `torch_sdpa`（稳）；SageAttention(sm120) 编上了自动用 `sage_attn2`（快）。

**runner** = `wan2.2_moe`（A14B 双专家），`cpu_offload` 按显存自动。**前端纯 t2v**：粘小说 → 一键分析 → 每镜「出片(t2v)」，无出图/选图。

In [ ]:
# === §1 参数 + Drive + GPU 探测 ===
import os, sys
DEEPSEEK_KEY = ''            # 分镜/AI 分析用;留空=用前端配的 grok
REPO_URL = 'https://github.com/aw3n1998/build-with-langchain.git'; BRANCH = 'main'
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN') or ''
except Exception:
    pass
if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'; CACHE = DRIVE + '/mirage_models'; TOOLS = DRIVE + '/mirage_tools'
for p in (CACHE, TOOLS):
    os.makedirs(p, exist_ok=True)
os.environ['HF_HOME'] = TOOLS + '/hf_cache'; os.makedirs(os.environ['HF_HOME'], exist_ok=True)
!nvidia-smi -L
import torch
if torch.cuda.is_available():
    gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    cc = torch.cuda.get_device_capability(0)
    os.environ['LX_CPU_OFFLOAD'] = 'false' if gb > 70 else 'true'   # >70G 两专家全 GPU;否则挪 CPU
    print(f'GPU {torch.cuda.get_device_name(0)} sm{cc[0]}.{cc[1]} {gb:.0f}G -> cpu_offload={os.environ["LX_CPU_OFFLOAD"]}')
    if cc[0] >= 12:
        print('  Blackwell(sm120): §3 编译 SageAttention(官方含 12.0)、patch worldmirror。')
else:
    os.environ['LX_CPU_OFFLOAD'] = 'true'; print('没检测到 GPU! 更改运行时类型 -> 选 GPU')
os.environ['PYTHONPATH'] = os.pathsep.join(p for p in sys.path if p)   # 子进程继承内核 torch
print('环境就绪 | 模型:', CACHE, '| 工具:', TOOLS)

In [ ]:
# === §2 拉取/更新 mirage 仓库 ===
import os, sys
%cd /content
if not os.path.isdir('mirage/.git'):
    !git clone -b {BRANCH} {REPO_URL} mirage
else:
    !cd mirage && git fetch origin {BRANCH} -q && git checkout {BRANCH} -q && git pull -q
%cd /content/mirage
sys.path.insert(0, '/content/mirage/colab')
print('mirage 就绪 @', BRANCH)

In [ ]:
# === §3 装 lightx2v（torch 锁 cu128 + --no-deps 绕 torch<2.8.0 冲突 + patch worldmirror/ring_attn）===
# 今日实战踩平的 5 坑(详见 git log / 笔记 lightx2v-t2v-blackwell-install):
#   ① lightx2v 0.1.0 钉 torch<2.8.0 撞 cu128-torch2.8 → 本体&依赖全 --no-deps 装(不解析 torch);
#   ② 残留元数据(pip 说装了却 import 不了)→ 装前 uninstall 清掉;③ editable 的 .pth 内核不重读 → sys.path 加源码;
#   ④ worldmirror/ring_attn 跟单卡 t2v 无关却崩 import → patch 文件(server 子进程也生效);⑤ torch↔tv ABI 不匹配 → 原子重装匹配套。
# 心法:一切以「干净子进程」import 为准(内核会被装卸污染;§5 server 本就是子进程)。
import os, subprocess, sys, json, re, importlib.util as u
def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True).stdout
def _ok(mod): return 'Traceback' not in sh(f'python -c "import {mod}" 2>&1')   # 子进程 import 成功=无 Traceback
LX = '/content/LightX2V'
# 1) clone + 源码加进 sys.path(editable 的 .pth 当前内核不重读)
if not os.path.isdir(LX + '/.git'):
    sh(f'git clone https://github.com/ModelTC/LightX2V.git {LX}')
if LX not in sys.path: sys.path.insert(0, LX)
# 2) torch 三件套:缺了或 torchvision ABI 坏了(子进程探测)→ 一条命令原子重装匹配的 cu128 套
if not _ok('torchvision'):
    print('torch/torchvision 缺或 ABI 不匹配 → 原子重装匹配的 cu128 三件套...')
    print(sh('pip install -q --force-reinstall --no-deps torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 2>&1 | tail -3'))
v = sh("python -c 'import torch,torchvision,torchaudio;print(torch.__version__);print(torchvision.__version__);print(torchaudio.__version__)'").split()
con = '/content/torch_con.txt'; open(con, 'w').write('\n'.join(f'{p}=={x}' for p, x in zip(('torch', 'torchvision', 'torchaudio'), v[:3])))
print('torch 锁定:', v)
# 3) patch 跟单卡 t2v 无关、却崩 import 的两处(改文件→server 子进程也生效;幂等)
fa = LX + '/lightx2v/models/networks/worldmirror/models/layers/attention.py'   # worldmirror 硬 import flash_attn
if os.path.exists(fa):
    s = open(fa).read()
    s = s.replace('from flash_attn_interface import flash_attn_func as flash_attn_func_v3', 'flash_attn_func_v3 = None  # [patch]')
    s = s.replace('from flash_attn.flash_attn_interface import flash_attn_func as flash_attn_func_v2', 'flash_attn_func_v2 = None  # [patch]')
    open(fa, 'w').write(s)
fr = LX + '/lightx2v/common/ops/attn/ring_attn.py'   # ring_attn(多卡分布式,单卡不用)的 @torch.jit.script 新 torch 编不过
if os.path.exists(fr):
    s = open(fr).read()
    s = re.sub(r'@torch\.jit\.script', '# @torch.jit.script  # [patch]', s)
    s = re.sub(r'torch\.jit\.script\(\s*([A-Za-z_]\w*)\s*\)', r'\1', s)
    open(fr, 'w').write(s)
# 4) 装 lightx2v(子进程 import 不过才装):本体 --no-deps 绕 torch<2.8.0 冲突;依赖也 --no-deps 装(不解析子依赖)
if not _ok('lightx2v'):
    sh('pip uninstall -y lightx2v')   # 清残留,别用 PyPI 残壳
    print('装 lightx2v 本体(--no-deps,绕 torch<2.8.0 冲突)...'); print(sh(f'cd {LX} && pip install -e . --no-deps 2>&1 | tail -3'))
    import importlib.metadata as md
    try: reqs = md.requires('lightx2v') or []
    except Exception: reqs = []
    deps = [r.split(';')[0].strip() for r in reqs if 'extra ==' not in r and 'extra==' not in r]
    deps = [d for d in deps if d and not d.lower().startswith('torch')]   # 去 torch(用现成 cu128)
    open('/content/lx_deps.txt', 'w').write('\n'.join(deps))
    print(f'补装依赖({len(deps)} 个,--no-deps -c con 不碰 torch)...'); print(sh(f'pip install -r /content/lx_deps.txt --no-deps -c {con} 2>&1 | tail -3'))
    NAME = {'cv2': 'opencv-python', 'PIL': 'pillow', 'yaml': 'pyyaml', 'sklearn': 'scikit-learn', 'skimage': 'scikit-image'}
    for _ in range(15):   # 子进程 import 缺啥补啥
        err = sh('python -c "import lightx2v" 2>&1')
        if 'Traceback' not in err: break
        m = re.search(r"No module named '([\w.]+)'", err)
        if not m: print('⚠ import 非缺包错:', err[-300:]); break
        mod = m.group(1).split('.')[0]; pkg = NAME.get(mod, mod); print('缺', mod, '→装', pkg); sh(f'pip install {pkg} --no-deps -c {con}')
# 5) SageAttention(可选提速,官方含 12.0=sm120;失败自动退 torch_sdpa,不影响出片)
if not _ok('sageattention'):
    if not os.path.isdir('/content/SageAttention/.git'):
        sh('git clone https://github.com/thu-ml/SageAttention.git /content/SageAttention')
    print('编译 SageAttention(sm120,约 3-8 分钟;失败退 torch_sdpa)...')
    print(sh(f'cd /content/SageAttention && CUDA_ARCHITECTURES="8.0,8.6,8.9,9.0,12.0" EXT_PARALLEL=4 '
             f'NVCC_APPEND_FLAGS="--threads 8" MAX_JOBS=32 pip install -e . --no-deps -c {con} 2>&1 | tail -5'))
_sage = _ok('sageattention')
# 6) config:注意力算子 + 显存策略(★兼容 5090 32G:小卡 cpu_offload+模型级粒度,lightx2v 官方称 24G 即够、32G 余量更大)
cfg = LX + '/configs/wan/wan_t2v.json'; c = json.load(open(cfg))
_attn = 'sage_attn2' if _sage else 'torch_sdpa'
for k in ('self_attn_1_type', 'cross_attn_1_type', 'cross_attn_2_type'): c[k] = _attn
_off = (os.environ.get('LX_CPU_OFFLOAD', 'true') == 'true')   # §1 按显存:>70G=False(双专家全GPU) / ≤70G=True(卸载)
c['cpu_offload'] = _off
if _off:   # 小卡(5090 32G 等):cpu_offload + 模型级粒度 + 不卸载 t5/vae;仍 OOM 再把 offload_granularity 改 'block'→'phase'
    c['offload_granularity'] = 'model'; c['offload_ratio'] = 1.0
    c['t5_cpu_offload'] = False; c['vae_cpu_offload'] = False
else:      # 大卡(>70G):双专家全 GPU bf16 不卸载(最快)
    c['offload_granularity'] = 'block'
# fp8 量化(可选,默认关):5090 sm120 原生 fp8 更省更快,但 lightx2v 没有现成 T2V-A14B fp8 权重 →
#   需 tools/convert/converter.py --linear_type fp8 自转 high/low 双专家 + 装 sgl-kernel,且 cu128/sm120 待真机验证。
#   确认转好后开启:c['dit_quantized']=True; c['dit_quant_scheme']='fp8-sgl'; c['dit_quantized_ckpt']='<fp8权重目录>'
json.dump(c, open(cfg, 'w'), indent=4)
# 7) 终检(干净子进程为准=§5 server 启动环境)
print('—' * 10)
print('lightx2v import:', '✅ OK' if _ok('lightx2v') else '❌ 看上面报错', '| SageAttention:', _sage,
      '| attn:', _attn, '| cpu_offload:', _off, ('(小卡:model级卸载,5090 友好)' if _off else '(大卡:双专家全GPU)'))

In [ ]:
# === §4 下权重到本地 /content（直接从 HF 下到本地 SSD：比 Drive FUSE 读快几倍 + 本地加载秒级）===
# 取舍(用户定):本地盘每会话重下 ~67G(HF CDN 快,~10 分钟),换加载快、无 Drive FUSE 卡顿。
#   想持久省流量→把 MODEL 指回 Drive 的 CACHE 路径(但加载会慢)。LoRA 小,仍放 Drive 持久。
import os, glob
from huggingface_hub import snapshot_download
MODEL = '/content/wan_local'; LORA = CACHE + '/lightx2v_t2v_lora'
os.makedirs(MODEL, exist_ok=True); os.makedirs(LORA, exist_ok=True)
_have = (len(glob.glob(MODEL + '/high_noise_model/*.safetensors')) == 6 and len(glob.glob(MODEL + '/low_noise_model/*.safetensors')) == 6)
if not _have:
    print('从 HF 下 Wan2.2-T2V-A14B 到本地(~67G,并行8线程,断点续传)...')
    snapshot_download('Wan-AI/Wan2.2-T2V-A14B', local_dir=MODEL, max_workers=8,
        allow_patterns=['high_noise_model/*', 'low_noise_model/*', '*.json', 'Wan2.1_VAE.pth', 'models_t5_umt5-xxl-enc-bf16.pth', 'google/*'])
else:
    print('本地已有完整权重,跳过下载。')
# 4步蒸馏 LoRA(小)→Drive 持久
_LORA = 'Wan2.2-T2V-A14B-4steps-lora-rank64-Seko-V2.0'
snapshot_download('lightx2v/Wan2.2-Lightning', local_dir=LORA, allow_patterns=[f'{_LORA}/*'])
hi = len(glob.glob(f'{MODEL}/high_noise_model/*.safetensors')); lo = len(glob.glob(f'{MODEL}/low_noise_model/*.safetensors'))
os.environ['LIGHTX2V_MODEL_T2V'] = MODEL
print('本地权重 high/low:', hi, '/', lo, '✅' if (hi == 6 and lo == 6) else '❌ 缺→重跑本格(续传)', '| MODEL =', MODEL)

In [ ]:
# === §5 起 lightx2v server（model_cls=wan2.2_moe；端口 8189；脱离内核,停 cell 不杀）===
import os, subprocess, sys, glob
sys.path.insert(0, '/content/mirage/colab'); import persist
LX = '/content/LightX2V'
# 权重源:优先本地(§4b 拷过=秒载),否则退 Drive(慢)
_local = '/content/wan_local'
MODEL = _local if (len(glob.glob(_local + '/high_noise_model/*.safetensors')) == 6 and len(glob.glob(_local + '/low_noise_model/*.safetensors')) == 6) else CACHE + '/wan2.2_t2v_a14b'
_isloc = (MODEL == _local)
subprocess.run("fuser -k 8189/tcp 2>/dev/null; pkill -9 -f 'lightx2v.server' 2>/dev/null; true", shell=True)  # 硬重启
e = dict(os.environ); e['CUDA_VISIBLE_DEVICES'] = '0'; e['PYTHONUNBUFFERED'] = '1'
open('/content/lightx2v.log', 'w').close()
subprocess.Popen(['python', '-u', '-m', 'lightx2v.server', '--model_cls', 'wan2.2_moe', '--task', 't2v',
                  '--model_path', MODEL, '--config_json', f'{LX}/configs/wan/wan_t2v.json',
                  '--host', '0.0.0.0', '--port', '8189'],
                 cwd=LX, env=e, stdout=open('/content/lightx2v.log', 'w'), stderr=subprocess.STDOUT, start_new_session=True)
print('lightx2v server 起中 | 权重源:', '本地 SSD(秒载) ✅' if _isloc else 'Drive(慢,最多 15 分钟;建议先跑 §4b 拷本地再回本格)')
ok = persist.wait_http('http://127.0.0.1:8189/v1/service/status', timeout=(240 if _isloc else 900))
print('✅ lightx2v 就绪' if ok else '✗ 还没就绪 → 别重跑本格(会重载)!跑 §5c 查进度;Drive 慢就先 §4b 拷本地再回 §5')
print(subprocess.run('tail -40 /content/lightx2v.log', shell=True, capture_output=True, text=True).stdout)
print('\n★只有崩在 KeyError patch_embedding 才跑 §5b;否则就是慢,等它或跑 §4b 拷本地。')

In [ ]:
# === §5b (仅当 §5 崩在「KeyError patch_embedding / 加载权重」时跑) diffusers→native 转格式 ===
import glob
LX = '/content/LightX2V'; MODEL = CACHE + '/wan2.2_t2v_a14b'
for d in ('high_noise_model', 'low_noise_model'):
    !cd {LX} && python tools/convert/converter.py --source {MODEL}/{d}/ --output {MODEL}/{d}/ --output_ext .safetensors --output_name wan_{d} --model_type wan_dit --single_file
f = glob.glob(f'{MODEL}/high_noise_model/wan_*.safetensors')
if f:
    from safetensors import safe_open
    print('转后 key 样例:', list(safe_open(f[0], 'pt').keys())[:5])
print('转完 → 回 §5 重起 server')

In [ ]:
# === §5c 查 server 状态（不重启！server 加载 67G 慢,用这个看进度,别重跑 §5）===
import subprocess
def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True).stdout
alive = bool(sh("pgrep -f 'lightx2v.server'").strip())
st = sh("curl -s -m 5 http://127.0.0.1:8189/v1/service/status") or '(还没响应,可能仍在加载)'
print('① server 进程:', '活着 ✅' if alive else '没了 ❌(崩了→看下面日志尾有无 Traceback)')
print('② status 端点:', st)
print('③ 日志尾 30 行:')
print(sh('tail -30 /content/lightx2v.log'))
print('\n判断: 进程活+status有响应=就绪(去 §6/§8/§9);进程活+无响应+日志在动无Traceback=还在加载(等几分钟再跑本格);进程没了+有Traceback=崩了。')

In [ ]:
# === §6 装后端依赖 + 写 .env（纯 t2v：COMFYUI 空、lightx2v 接通）===
import os, re, shutil
%cd /content/mirage
!pip -q install -r requirements.txt
!pip -q install fastembed edge-tts
shutil.copy('.env.colab', '.env'); env = open('.env').read()
def _se(e, k, v): return re.sub(k + r'=.*', k + '=' + v, e) if (k + '=') in e else e + ('' if e.endswith(chr(10)) else chr(10)) + k + '=' + v + chr(10)
MODEL = CACHE + '/wan2.2_t2v_a14b'
_LD = CACHE + '/lightx2v_t2v_lora/Wan2.2-T2V-A14B-4steps-lora-rank64-Seko-V2.0'
for k, v in [('COMFYUI_BASE_URL', ''), ('LIGHTX2V_ENABLED', 'true'), ('LIGHTX2V_BASE_URL', 'http://127.0.0.1:8189'),
             ('LIGHTX2V_MODEL_T2V', MODEL), ('T2V_PROVIDER', 'lightx2v-t2v'), ('VIDEO_PROVIDER_DEFAULT', 'wan2.2'),
             ('WAN_T2V_LORA_HIGH', _LD + '/high_noise_model.safetensors'),
             ('WAN_T2V_LORA_LOW', _LD + '/low_noise_model.safetensors')]:
    env = _se(env, k, v)
if DEEPSEEK_KEY:
    env = _se(env, 'OPENAI_API_KEY', DEEPSEEK_KEY)
open('.env', 'w').write(env)
print('.env 写好(纯 t2v: COMFYUI 空 / lightx2v 接通 / 蒸馏 LoRA 登记)')

In [ ]:
# === §7 构建前端（纯 t2v 工作台）===
%cd /content/mirage/frontend
!npm install --silent && npm run build || echo '前端构建失败(用 API 也行)'
%cd /content/mirage

In [ ]:
# === §8 起后端（硬重启：杀旧+起新，读 §6 的 .env）===
import sys; sys.path.insert(0, '/content/mirage/colab'); import persist
persist.restart_service('后端', ['uvicorn', 'mirage.main_api:app', '--host', '0.0.0.0', '--port', '8000'],
    'http://127.0.0.1:8000/api/health', '/content/api.log', 8000, 'uvicorn', cwd='/content/mirage', timeout=120)

In [ ]:
# === §9 cloudflared 暴露后端 → 公网 URL ===
import subprocess, re, time, os, sys
sys.path.insert(0, '/content/mirage/colab'); import persist
subprocess.run('pkill -9 -f cloudflared 2>/dev/null; sleep 2', shell=True)
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared', shell=True)
open('/content/cf.log', 'w').close()
persist.start_bg(['cloudflared', 'tunnel', '--url', 'http://localhost:8000'], '/content/cf.log')
url = None
for _ in range(60):
    time.sleep(1)
    m = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/cf.log').read())
    if m:
        url = m[-1]; break
print('✅ 公网地址:', url if url else '⚠ 60s 未就绪,重跑本格')
print('打开它 → 工作台纯 t2v: 粘小说 → 一键分析 → 每镜「出片(t2v)」')

In [ ]:
# === §10 实时日志（lightx2v 出片采样进度；停本格只停 tail、不杀 server）===
!tail -n 40 -f /content/lightx2v.log

---
## 角色 LoRA 训练（Wan-T2V，可选 · 独立）
训练用 `colab_deploy.ipynb` 的 **LW1/LW2**（ai-toolkit，与 lightx2v 推理两套，互不影响）。训好的 `角色_wan_t2v_high/low.safetensors` 放 Drive，§6 的 `.env` 里把 `WAN_T2V_LORA_HIGH/LOW` 指过去即可（触发词也在项目里填）。

## 首跑核对清单
1. **§1** 打印 `GPU ... smX.Y ...G -> cpu_offload=...`？没 GPU → 运行时改选 GPU。
2. **§3** 末行 `lightx2v import: ✅ OK`？（`SageAttention: False` 不影响出片，走 `torch_sdpa` 略慢）
3. **§4** `权重 high/low: 6 / 6 ✅`？否 → 重跑 §4（`snapshot_download` 断点续传，只补缺的）。
4. **§5** `✅ lightx2v 就绪`？崩 `KeyError patch_embedding / 加载权重` → 跑 §5b 转格式，再回 §5。
5. **§6 → §8 → §9** .env 接通 lightx2v → 后端起 → 公网 URL → 打开工作台出片。

## 排错速记（2026-06-17 实战）
- 任何 `import` 报错先看是不是**内核被装卸污染**——重开 §3 用子进程判据会自愈；实在乱了就「代码执行程序 → 重启」后 Run all。
- `ResolutionImpossible / torch 被降级` → 是 lightx2v 钉 `torch<2.8.0`，§3 已用 `--no-deps` 绕过，别手动 `pip install lightx2v`（PyPI 是残壳）。
- 详细 5 坑 5 解见提交记录与笔记 `lightx2v-t2v-blackwell-install`。